In [ ]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ================= CONFIGURATION =================
INPUT_CSV = "final_news_sentiment_analysis.csv" 
OUTPUT_CSV = "sentiment_data.csv"

# Updated Asset List based on what is actually in your file
# We use the symbols as they appear in the CSV (No '.NS')
ASSETS = ["RELIANCE", "TCS", "INFY", "ICICIBANK", "ITC"]

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using Device: {'GPU' if DEVICE==0 else 'CPU'}")

# ================= 1. LOAD & CLEAN DATA =================
print(f"Loading {INPUT_CSV}...")

try:
    df = pd.read_csv(INPUT_CSV)
    
    # 1. Normalize Column Names
    # Your file has 'Publish Date', 'Headline', 'Symbol'
    df.rename(columns={
        'Publish Date': 'date', 
        'Headline': 'headline',
        'Symbol': 'symbol'
    }, inplace=True)
    
    # 2. Filter for our Portfolio
    # We filter by SYMBOL because your file has a 'Symbol' column!
    df = df[df['symbol'].isin(ASSETS)].copy()
    
    print(f"Filtered down to {len(df)} headlines for {ASSETS}")

except FileNotFoundError:
    print("Error: File not found. Make sure you uploaded the CSV to Kaggle!")
    exit()

# ================= 2. FINBERT SCORING =================
print("Loading FinBERT Model...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=DEVICE, truncation=True, max_length=512)

print("Scoring headlines...")

batch_size = 64
headlines = df['headline'].astype(str).tolist()
results = []

# Process in batches
for i in tqdm(range(0, len(headlines), batch_size)):
    batch = headlines[i:i+batch_size]
    try:
        outputs = pipe(batch)
        for out in outputs:
            score = out['score']
            if out['label'] == 'negative':
                score *= -1
            elif out['label'] == 'neutral':
                score = 0
            results.append(score)
    except Exception as e:
        print(f"Batch error: {e}")
        results.extend([0.0] * len(batch))

df['finbert_score'] = results

# ================= 3. AGGREGATE & SMOOTH =================
print("Aggregating to Daily Sentiment...")

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df.dropna(subset=['date'], inplace=True)

# Group by Date -> Average Score
daily_sentiment = df.groupby('date')['finbert_score'].mean().reset_index()
daily_sentiment.rename(columns={'finbert_score': 'sentiment_score'}, inplace=True)

# Apply Smoothing
daily_sentiment['sentiment_smooth'] = daily_sentiment['sentiment_score'].ewm(span=5).mean()

# ================= 4. SAVE =================
daily_sentiment.to_csv(OUTPUT_CSV, index=False)
print(f"DONE! Saved {len(daily_sentiment)} days of sentiment to {OUTPUT_CSV}")
print(daily_sentiment.tail())